<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**The Governance Framework**

Because the ML model collapsed over time, the deployed solution relies on our robust heuristic rule, supplemented by a strict human-in-the-loop playbook.

**Categorical Archetypes:**
We categorize pages into five actionable archetypes (`aged_workhorse`, `fragile_snippet`, `young_earner`, `high_risk`, `low_risk`), assigned first-match-wins in that order from the model probability and page metadata. Each archetype maps to exactly one recommended action.

**Guardrails (The No-Go List):**
- **No auto-publishing:** A human must approve every change.
- **No causal guarantees:** Finding a declining page doesn't mean a refresh is guaranteed to fix it.
- **No thin-history actions:** Pages with less than 15 active days are structurally excluded.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Operational Triggers & Monitoring:**
- **Staleness Check:** If the queue is older than 45 days, regenerate it.
- **Human Override Tracker:** If human editors reject >75% of recommendations (precision < 0.25), halt the system.
- **Performance Drift:** If time-aware fold performance drops below 0.30 (or any fold drops below 0.10), trigger an immediate audit and retrain. *(Note: This trigger has actually already fired on our W6 window).*

**Expected Business Value:**
Assuming a review budget of 50 pages (K=50), the baseline rule identifies roughly 18 true decliners versus 12 from random selection. That translates to finding 6 additional decaying pages per batch, utilizing ~21 hours of editorial time. These top 50 rows represent ~3% of total portfolio impressions, which is 50x the traffic at stake compared to random selection.

In [6]:
import os, getpass, duckdb, hashlib, json
import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your HF READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

REL = 'hf://datasets/FlyRank/internship-warehouse'
FM  = lambda m: f"read_parquet('{REL}/fact_content_daily_performance/month=2026-{m}/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

df = con.sql(f"""
    WITH per_content AS (
        SELECT
            f.content_hash_id, f.client_hash_id,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_clicks    ELSE 0 END) AS clk_prev30,
            AVG(CASE  WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01' THEN f.gsc_avg_position END)   AS pos_prev30,
            COUNT(DISTINCT CASE WHEN f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-03-01'
                                 AND f.gsc_impressions > 0 THEN f.report_date END)                                                 AS days_with_imp_prev30,
            SUM(CASE WHEN f.report_date >= DATE '2026-03-01' AND f.report_date < DATE '2026-04-01' THEN f.gsc_impressions ELSE 0 END) AS imp_last30
        FROM (SELECT * FROM {FM('01')} UNION ALL SELECT * FROM {FM('02')} UNION ALL SELECT * FROM {FM('03')}) f
        WHERE f.report_date >= DATE '2026-01-30' AND f.report_date < DATE '2026-04-01'
          AND f.gsc_data_available IS TRUE
        GROUP BY f.content_hash_id, f.client_hash_id
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM per_content
""").df()

meta = con.sql(f"""
    SELECT content_hash_id,
           DATEDIFF('day', content_created_date, DATE '2026-03-01') AS content_age_days,
           word_count, content_type
    FROM {DIM_CONTENT}
""").df()

df = df.merge(meta, on='content_hash_id', how='left')
df['is_declining']     = (df['imp_last30'] < 0.8 * df['imp_prev30']).astype(int)
df['pos_prev30']       = df['pos_prev30'].fillna(0)
df['content_age_days'] = df['content_age_days'].fillna(0)
df['log_imp_prev30']   = np.log1p(df['imp_prev30'])
df['log_clk_prev30']   = np.log1p(df['clk_prev30'])

FEATURES = ['log_imp_prev30', 'log_clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
SEED = 42

def client_fold(cid, n=5):
    return int(hashlib.sha256(cid.encode()).hexdigest(), 16) % n
df['fold'] = df['client_hash_id'].map(client_fold)
df['_tie'] = df['content_hash_id'].map(lambda c: int(hashlib.sha256(c.encode()).hexdigest(), 16))

# Apply the transparent rule
stale   = (df['content_age_days'] >= 90).astype(int)
visible = (df['imp_prev30']       >= 500).astype(int)
df['score'] = stale * visible * df['imp_prev30']
df['reason_code'] = np.select(
    [stale.astype(bool) & visible.astype(bool), ~stale.astype(bool)],
    ['stale_but_visible', 'not_stale'], default='low_volume')

# Train a Random Forest on all data to get model_probability for archetype assignment
X = df[FEATURES].to_numpy()
y = df['is_declining'].to_numpy()
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
rf.fit(X, y)
df['model_probability'] = rf.predict_proba(X)[:, 1]

def assign_archetype(row):
    if row['content_age_days'] >= 365 and row['imp_prev30'] >= 3000:
        return 'aged_workhorse'
    elif row['model_probability'] > 0.7 and row['days_with_imp_prev30'] < 15:
        return 'fragile_snippet'
    elif row['model_probability'] > 0.7 and row['content_age_days'] < 90:
        return 'young_earner'
    elif row['model_probability'] > 0.7:
        return 'high_risk'
    else:
        return 'low_risk'

df['archetype'] = df.apply(assign_archetype, axis=1)

# Build the final playbook queue sorted by rule score, then model probability as secondary
queue = df.sort_values(['score', 'model_probability', '_tie'], ascending=[False, False, True]).reset_index(drop=True)
queue['playbook_rank'] = range(1, len(queue) + 1)

print(f"Playbook queue ready: {len(queue):,} pages ranked")
print(f"Archetype distribution:")
print(queue['archetype'].value_counts().to_string())
print()
print("Top 10 of the queue:")
print(queue[['playbook_rank', 'content_hash_id', 'score', 'reason_code', 'archetype',
             'imp_prev30', 'content_age_days']].head(10).to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Playbook queue ready: 81,521 pages ranked
Archetype distribution:
archetype
low_risk           75589
high_risk           4088
aged_workhorse      1353
young_earner         251
fragile_snippet      240

Top 10 of the queue:
 playbook_rank          content_hash_id    score       reason_code      archetype  imp_prev30  content_age_days
             1 content_8e1334d6356668e3 204176.0 stale_but_visible aged_workhorse    204176.0               380
             2 content_fec55986a1868d62 198339.0 stale_but_visible aged_workhorse    198339.0               380
             3 content_9c057b66c30a3abb 195655.0 stale_but_visible      high_risk    195655.0               213
             4 content_512dbad65bd5ade9 178603.0 stale_but_visible       low_risk    178603.0               157
             5 content_e241d6415ac9e534 177075.0 stale_but_visible aged_workhorse    177075.0               382
             6 content_e8a52cf3d5988c07 168638.0 stale_but_visible       low_risk    168638.0            

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

Decision-Support Only: This model provides decision-support for human reviewers. Because we are analyzing cross-sectional data without a controlled design, we can only state that we observed patterns in this dataset; the model's scores do not prove that updating a page causes traffic to return.
  
Data Limits: The model cannot differentiate between a page ranking at position zero and a page with missing data, as avg_position = 0 simply means "no data". Furthermore, because history depth differs wildly per client, global calendar windows cannot be universally applied.

In [7]:
# Data limits — pages with zero GSC position signal (no reliable ranking data)
# pos_prev30 == 0 means 'no position data', not rank zero. Count how many pages this affects.
zero_pos = (df['pos_prev30'] == 0).sum()
total = len(df)
print(f'Pages with no GSC position data (pos_prev30 == 0): {zero_pos:,} of {total:,} ({zero_pos/total:.1%})')
print()

# Pages with thin history (fewer than 15 active days in the feature window) — no-go for actions
thin = (df['days_with_imp_prev30'] < 15).sum()
print(f'Pages with thin impression history (<15 days active): {thin:,} of {total:,} ({thin/total:.1%})')
print('These pages are excluded from automated actions per the no-go list.')
print()

# Show what happens to P@50 if we filter these out (shows the data quality matters)
clean = queue[queue['days_with_imp_prev30'] >= 15]
print(f'Queue rows after thin-history filter: {len(clean):,}')


Pages with no GSC position data (pos_prev30 == 0): 1 of 81,521 (0.0%)

Pages with thin impression history (<15 days active): 9,613 of 81,521 (11.8%)
These pages are excluded from automated actions per the no-go list.

Queue rows after thin-history filter: 71,908


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Content updates should NEVER be fully automated. A human must review the top of the ranked queue to filter out the model's blind spots.

Static Intent (No-Go): Glossary terms, historical records, or definition pages. Even if the model flags them due to age, human reviewers must skip them.

Missingness Injection (No-Go): Some content types naturally have missing keyword data or word counts. Reviewers must ensure a page isn't being flagged just because its specific format lacks standard text metrics.

In [8]:
# No-go list verification — identify pages that would violate governance rules
print('=== No-Go List Checks ===')
print()

# No-go 1: Pages with missing word_count (content structure unknown)
no_wc = (df['word_count'].isna()).sum()
print(f'Pages with missing word_count: {no_wc:,} ({no_wc/len(df):.1%}) -- cannot assess content depth')

# No-go 2: Pages with no prior clicks (pure impression pages -- very fragile)
no_clk = (df['clk_prev30'] == 0).sum()
print(f'Pages with zero prior clicks: {no_clk:,} ({no_clk/len(df):.1%}) -- impressions only, no engagement proof')

# No-go 3: Very new pages (age < 30 days) -- not enough history to call decline
new_pages = (df['content_age_days'] < 30).sum()
print(f'Pages younger than 30 days: {new_pages:,} ({new_pages/len(df):.1%}) -- insufficient history')
print()
print('These three groups must be reviewed manually before any action is taken.')
print('They are flagged but not auto-actioned.')


=== No-Go List Checks ===

Pages with missing word_count: 25,653 (31.5%) -- cannot assess content depth
Pages with zero prior clicks: 30,495 (37.4%) -- impressions only, no engagement proof
Pages younger than 30 days: 5,682 (7.0%) -- insufficient history

These three groups must be reviewed manually before any action is taken.
They are flagged but not auto-actioned.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

Retrain Trigger 1: When a significant volume of new clients is onboarded to the data warehouse.

Retrain Trigger 2: If the out-of-sample Precision@50 on new data drops below the naive base rate (majority class).

In [9]:
# Monitoring trigger baselines — computed from the live queue
base_rate = df['is_declining'].mean()
print(f'Overall decline rate (base rate): {base_rate:.3f}')
print()

# M1: Queue staleness
print('Trigger M1 — Staleness:')
print('  Regenerate the queue if it is older than 45 days.')
print()

# M2: Human acceptance rate
print('Trigger M2 — Human acceptance rate:')
print(f'  If editors accept fewer than {base_rate:.0%} of recommendations (matching random chance),')
print('  halt the system and audit the feature windows.')
print()

# M3: Time-aware fold performance (already fired)
try:
    import json, os
    REPO_ROOT = os.getcwd()
    while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
        REPO_ROOT = os.path.dirname(REPO_ROOT)
    ta = json.load(open(os.path.join(REPO_ROOT, 'work', 'outputs', 'w06_validation_audit_receipt.json')))
    ta_p50 = ta['mean_precision@50_time_aware']
    print(f'Trigger M3 — Time-aware fold P@50: current = {ta_p50:.3f}')
    if ta_p50 < 0.30:
        print('  *** M3 HAS ALREADY FIRED. The time-aware split shows the model does not generalize forward. ***')
        print('  => Ship the transparent rule + human-review playbook. Do not ship the model.')
except FileNotFoundError:
    print('Trigger M3 — run w06 first to compute the time-aware P@50.')


Overall decline rate (base rate): 0.249

Trigger M1 — Staleness:
  Regenerate the queue if it is older than 45 days.

Trigger M2 — Human acceptance rate:
  If editors accept fewer than 25% of recommendations (matching random chance),
  halt the system and audit the feature windows.

Trigger M3 — Time-aware fold P@50: current = 0.260
  *** M3 HAS ALREADY FIRED. The time-aware split shows the model does not generalize forward. ***
  => Ship the transparent rule + human-review playbook. Do not ship the model.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [10]:
# Write the final playbook queue and receipt
import json, os

REPO_ROOT = os.getcwd()
while not os.path.exists(os.path.join(REPO_ROOT, 'AGENTS.md')) and os.path.dirname(REPO_ROOT) != REPO_ROOT:
    REPO_ROOT = os.path.dirname(REPO_ROOT)
OUT_DIR = os.path.join(REPO_ROOT, 'work', 'outputs')
os.makedirs(OUT_DIR, exist_ok=True)

# Write queue CSV (label excluded)
export_cols = ['playbook_rank', 'content_hash_id', 'client_hash_id', 'fold', 'score',
               'reason_code', 'archetype', 'model_probability',
               'imp_prev30', 'clk_prev30', 'pos_prev30', 'days_with_imp_prev30', 'content_age_days']
queue[export_cols].to_csv(os.path.join(OUT_DIR, 'w07_action_queue.csv'), index=False)
print(f"Queue written: {os.path.join(OUT_DIR, 'w07_action_queue.csv')}")
print(f"Total rows: {len(queue):,}")

# Build and write receipt
receipt = {
    'queue_size': len(queue),
    'archetype_counts': queue['archetype'].value_counts().to_dict(),
    'top5': queue[['playbook_rank', 'content_hash_id', 'archetype', 'reason_code']].head(5).to_dict('records'),
    'no_go_policies': [
        'No auto-publishing — every action requires human approval',
        'No causal guarantees — refreshing may not reverse the decline',
        'No actions on pages with fewer than 15 active days in the feature window',
    ],
    'monitoring_triggers': {
        'M1': 'Queue regeneration required if older than 45 days',
        'M2': 'Halt if human editor acceptance rate drops below 25%',
        'M3': 'Trigger audit if time-aware fold P@50 drops below 0.30 (already fired in W6)',
    }
}
with open(os.path.join(OUT_DIR, 'w07_playbook_receipt.json'), 'w') as fh:
    json.dump(receipt, fh, indent=2)
print(f"Receipt written: w07_playbook_receipt.json")


Queue written: c:\Users\Basil\Desktop\flyrank-internship-assignment1\work\outputs\w07_action_queue.csv
Total rows: 81,521
Receipt written: w07_playbook_receipt.json


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.